In [1]:
import pandas as pd
import pickle
import os
def load_community(num):
    graph = {}
    try:
        data = pd.read_csv(f"addhealth_edges/comm_{num:02d}.csv")
        for row in data.itertuples(index=False):
            source = int(row.source)
            target = int(row.target)
            if source not in graph:
                graph[source] = set()
            if target not in graph:
                graph[target] = set()
            graph[source].add(target)
            graph[target].add(source)
    except FileNotFoundError:
        pass
    return graph

def load_all_communities():
    all_graphs = {}
    for i in range(100):
        graph = load_community(i)
        if graph:
            all_graphs[i] = graph
    print(f"Loaded {len(all_graphs)} communities.")
    return all_graphs
graphs = load_all_communities()

if not os.path.exists('data/real_networks.pkl'):
    with open('data/real_networks.pkl', 'wb') as f:
        pickle.dump(graphs, f)

del graphs

Loaded 84 communities.


In [2]:
#BA network generation
import random
from tqdm import tqdm
def generate_ba_network(num_nodes, m):
    graph = {}
    for i in range(m):
        graph[i] = set()

    repeated = []
    for i in range(m):
        for j in range(i):
            graph[i].add(j)
            graph[j].add(i)
            repeated.append(i)
            repeated.append(j)

    for i in range(m, num_nodes):
        graph[i] = set()
        targets = set()
        while len(targets) < m:
            targets.add(random.choice(repeated))
        for t in targets:
            graph[i].add(t)
            graph[t].add(i)
            repeated.append(t)
            repeated.append(i)
    return graph

def save_ba_networks(m_list, size_list, count):
    for m in m_list:
        for size in size_list:
            print(f"Saving BA networks with m={m} and size={size}K")
            if not os.path.exists(f'data/ba/ba_network_m{m:02d}_N{size:03d}K.pkl'):
                networks = []
                for _ in tqdm(range(count)):
                    network = generate_ba_network(size*1000, m)
                    networks.append(network)
                with open(f'data/ba/ba_network_m{m:02d}_N{size:03d}K.pkl', 'wb') as f:
                    pickle.dump(networks, f)
                del networks

save_ba_networks(m_list = {3,5,7,9,11,13,15}, size_list = {10}, count = 500)
save_ba_networks(m_list = {5}, size_list = {20, 25, 30, 35, 40, 45, 50}, count = 250)
save_ba_networks(m_list = {3,5,7,9}, size_list = {1}, count = 500)


Saving BA networks with m=3 and size=10K


100%|██████████| 500/500 [00:17<00:00, 28.39it/s]


Saving BA networks with m=5 and size=10K


100%|██████████| 500/500 [00:27<00:00, 18.02it/s]


Saving BA networks with m=7 and size=10K


...

100%|██████████| 500/500 [00:01<00:00, 304.94it/s]


Saving BA networks with m=5 and size=1K


100%|██████████| 500/500 [00:02<00:00, 216.73it/s]


Saving BA networks with m=7 and size=1K


100%|██████████| 500/500 [00:03<00:00, 166.60it/s]


In [ ]:
from tqdm import tqdm
def generate_apollonian_network(generations):
    graph = {}
    def add_edge(u, v):
        graph[u].add(v)
        graph[v].add(u)

    graph[0] = set()
    graph[1] = set()
    graph[2] = set()
    add_edge(0, 1)
    add_edge(1, 2)
    add_edge(2, 0)

    next_node = 3
    faces = [(0, 1, 2)]
    for _ in tqdm(range(generations)):
        new_faces = []
        for a, b, c in faces:
            v = next_node
            next_node += 1
            graph[v] = set()

            add_edge(v, a)
            add_edge(v, b)
            add_edge(v, c)
            new_faces.extend([(a, b, v), (b, c, v), (c, a, v)])
        faces = new_faces
    return graph

for g in {8,9,11}:
    if not os.path.exists(f'data/apl/apl_network_g{g:02d}.pkl'):
        apl_network = generate_apollonian_network(g)
        with open(f'data/apl/apl_network_g{g:02d}.pkl', 'wb') as f:
            pickle.dump(apl_network, f)
        del apl_network

100%|██████████| 14/14 [00:05<00:00,  2.62it/s]


In [4]:
def generate_small_world_network(num_nodes, k, p):
    graph = {i: set() for i in range(num_nodes)}

    # Initial ring lattice
    for i in range(num_nodes):
        for j in range(1, k // 2 + 1):
            neighbor = (i + j) % num_nodes
            graph[i].add(neighbor)
            graph[neighbor].add(i)
    nodes = list(range(num_nodes))

    # Rewire
    for i in range(num_nodes):
        for j in range(1, k // 2 + 1):
            if random.random() > p:
                continue
            old_neighbor = (i + j) % num_nodes
            # Remove old edge
            graph[i].discard(old_neighbor)
            graph[old_neighbor].discard(i)
            # Avoid self-loops and duplicate edges
            forbidden = set(graph[i])
            forbidden.add(i)

            while True:
                new_neighbor = random.choice(nodes)
                if new_neighbor not in forbidden:
                    break

            graph[i].add(new_neighbor)
            graph[new_neighbor].add(i)
    return graph

def save_small_world_networks(p_list, size_list, count):
    for p in p_list:
        for size in size_list:
            print(f"Saving small-world networks with p={p} and size={size}K")
            if not os.path.exists(f'data/small_world/sw_network_p{p_list.index(p):02d}_N{size:03d}K.pkl'):
                networks = []
                for _ in tqdm(range(count)):
                    network = generate_small_world_network(size*1000, 4, p)
                    networks.append(network)
                with open(f'data/small_world/sw_network_p{p_list.index(p):02d}_N{size:03d}K.pkl', 'wb') as f:
                    pickle.dump(networks, f)
                del networks

import numpy as np
p_list = np.logspace(-4, 0, 20).tolist()
save_small_world_networks(p_list = p_list, size_list = {1,10}, count = 500)


Saving small-world networks with p=0.0001 and size=1K


100%|██████████| 500/500 [00:00<00:00, 941.32it/s] 


Saving small-world networks with p=0.0001 and size=10K


100%|██████████| 500/500 [00:06<00:00, 78.09it/s] 


Saving small-world networks with p=0.0001623776739188721 and size=1K


...

100%|██████████| 500/500 [00:17<00:00, 28.79it/s]


Saving small-world networks with p=1.0 and size=1K


100%|██████████| 500/500 [00:01<00:00, 312.07it/s]


Saving small-world networks with p=1.0 and size=10K


100%|██████████| 500/500 [00:20<00:00, 23.89it/s]


In [5]:
def generate_random_network(num_nodes, p):
    graph = {i: set() for i in range(num_nodes)}
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if random.random() <= p:
                graph[i].add(j)
                graph[j].add(i)
    return graph

def save_random_networks(p_list, size_list, count):
    for p in p_list:
        for size in size_list:
            print(f"Saving random networks with p={p} and size={size}K")
            if not os.path.exists(f'data/random/random_network_p{p_list.index(p):02d}_N{size:03d}K.pkl'):
                networks = []
                for _ in tqdm(range(count)):
                    network = generate_random_network(size*1000, p)
                    networks.append(network)
                with open(f'data/random/random_network_p{p_list.index(p):02d}_N{size:03d}K.pkl', 'wb') as f:
                    pickle.dump(networks, f)
                del networks

save_random_networks(p_list = [0.02, 0.04, 0.08], size_list = {1}, count = 500)

Saving random networks with p=0.02 and size=1K


100%|██████████| 500/500 [00:12<00:00, 38.76it/s]


Saving random networks with p=0.04 and size=1K


100%|██████████| 500/500 [00:14<00:00, 33.94it/s]


Saving random networks with p=0.08 and size=1K


100%|██████████| 500/500 [00:20<00:00, 24.95it/s]
